# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling
DuckDB `SUMMARIZE` dan transformasi Polars untuk memeriksa missing values,
outlier, duplikasi, serta validasi data dasar. Strategi cleaning disesuaikan
dengan karakteristik dataset Cosmopedia Indonesia.

Dataset yang digunakan adalah Cosmopedia Indonesia dengan 120.068 baris dan 9 kolom. Pemeriksaan difokuskan pada kolom inti `id`, `prompt`, dan `text`, serta metadata seperti `audience`, `format`, dan `seed_data`.

In [1]:
from pathlib import Path
import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cosmopedia_id.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "cosmopedia_id_clean.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. "
        "Pastikan Milestone 1 sudah menempatkan data di data/raw/."
    )

df = pl.read_csv(DATA_PATH, infer_schema_length=10_000)

print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
display(df.head())

Ukuran awal: 120,068 baris x 9 kolom


id,prompt,text,token_length,audience,format,seed_data,datasheet_global_id,created_at
i64,str,str,i64,str,str,str,i64,str
1,"""Berikut adalah ekstrak dari ha…","""Dahulu kala, ada sebuah agen y…",699,"""children""","""story""","""fineweb""",1,"""2026-02-18 07:35:54"""
24,"""Berikut kutipan dari sebuah ha…","""Suatu ketika, di sebuah negeri…",137,"""children""","""story""","""fineweb""",2,"""2026-02-18 08:19:23"""
25,"""Berikut kutipan dari sebuah ha…","""Di kota Popville, hiduplah ban…",93,"""children""","""story""","""fineweb""",3,"""2026-02-18 08:19:23"""
26,"""Berikut kutipan tentang chipmu…","""Di sebuah lingkungan yang tena…",76,"""children""","""story""","""fineweb""",4,"""2026-02-18 08:19:23"""
27,"""Berikut kutipan tentang terapi…","""Timmy pernah merasa sangat sed…",88,"""children""","""story""","""fineweb""",5,"""2026-02-18 08:19:23"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom, termasuk tipe data, persentase
nilai kosong, rentang nilai, dan jumlah nilai berbeda.

Pada dataset Cosmopedia Indonesia, profiling digunakan untuk memahami karakteristik kolom teks, metadata, dan panjang token sebelum proses pembersihan dilakukan.

In [2]:
connection = duckdb.connect()
connection.register("dataset", df)

profil = connection.execute(
    "SUMMARIZE SELECT * FROM dataset"
).fetchdf()

display(profil)

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,id,BIGINT,1,120091,117463,60057.49921711031,34660.791753253674,30138,59996,90002,120068,0.0
1,prompt,VARCHAR,"Berikut adalah cuplikan dari halaman web: ""Buk...",Write an educational story (3-5 paragraphs) ta...,114146,NaN,NaN,NaN,NaN,NaN,120068,0.0
2,text,VARCHAR,**1.3 Combinational Circuits**\n\nWelcome bac...,अगर आपको उchcha madhyamik standard mein हो तो ...,115976,NaN,NaN,NaN,NaN,NaN,120068,0.0
3,token_length,BIGINT,36,2787,1545,1187.9396508645102,194.61858027220862,1065,1168,1296,120068,0.0
4,audience,VARCHAR,,young_children,4,NaN,NaN,NaN,NaN,NaN,120068,0.0
5,format,VARCHAR,,textbook_unconditionned_topic,9,NaN,NaN,NaN,NaN,NaN,120068,0.0
6,seed_data,VARCHAR,,ultrachat,6,NaN,NaN,NaN,NaN,NaN,120068,0.0
7,datasheet_global_id,BIGINT,1,120068,117463,60034.5,34660.79039779672,30115,59973,89979,120068,0.0
8,created_at,VARCHAR,2026-02-18 07:35:54,2026-03-06 03:44:48,121032,NaN,NaN,NaN,NaN,NaN,120068,0.0


## 2. Deteksi Missing Values

Lihat jumlah dan persentase nilai kosong per kolom sebelum menentukan strategi
penanganannya. Pada dataset ini, `audience`, `format`, dan `seed_data` merupakan
kolom metadata yang memiliki nilai kosong.

Pemeriksaan dilakukan terhadap seluruh kolom dataset untuk mengetahui jumlah dan persentase nilai kosong. Kolom inti `id`, `prompt`, dan `text` menjadi perhatian utama karena digunakan untuk mengidentifikasi data serta merepresentasikan isi dokumen.

In [3]:
jumlah_baris = len(df)

null_summary = (
    df.null_count()
    .transpose(
        include_header=True,
        header_name="kolom",
        column_names=["jumlah_null"]
    )
    .with_columns(
        (
            pl.col("jumlah_null") / jumlah_baris * 100
        ).round(2).alias("persen_null")
    )
    .filter(pl.col("jumlah_null") > 0)
    .sort("jumlah_null", descending=True)
)

display(null_summary)

kolom,jumlah_null,persen_null
str,u32,f64


## 3. Penanganan Missing Values

Kolom `id`, `prompt`, dan `text` merupakan kolom inti. Baris yang tidak memiliki
salah satu nilai inti tersebut akan dikeluarkan. Nilai kosong pada metadata
(`audience`, `format`, dan `seed_data`) dipertahankan sebagai null karena tidak
tersedia dasar yang valid untuk menentukan nilai penggantinya.

Pada dataset Cosmopedia Indonesia, nilai kosong pada metadata tidak langsung diganti dengan kategori tertentu. Hal ini dilakukan untuk menghindari pemberian informasi yang tidak didukung oleh data asli.

In [4]:
# Normalisasi string kosong/berisi spasi pada kolom teks menjadi null.
kolom_teks = ["prompt", "text"]

df_normalized = df.with_columns([
    pl.when(pl.col(kolom).str.strip_chars() == "")
    .then(None)
    .otherwise(pl.col(kolom).str.strip_chars())
    .alias(kolom)
    for kolom in kolom_teks
])

# Hanya baris yang kehilangan kolom inti yang dihapus.
df_missing_handled = df_normalized.filter(
    pl.col("id").is_not_null()
    & pl.col("prompt").is_not_null()
    & pl.col("text").is_not_null()
)

print(f"Null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(
    "Null sesudah penanganan kolom inti: "
    f"{df_missing_handled.null_count().sum_horizontal().sum():,}"
)
print(f"Baris sebelum: {len(df):,}")
print(f"Baris sesudah: {len(df_missing_handled):,}")

Null sebelum: 0
Null sesudah penanganan kolom inti: 0
Baris sebelum: 120,068
Baris sesudah: 120,068


## 4. Deteksi dan Penanganan Outlier

Outlier pada `token_length` dideteksi menggunakan metode Interquartile Range
(IQR). Nilai yang terdeteksi tidak langsung dihapus atau diubah. Panjang token
yang tinggi dapat merepresentasikan teks yang memang lebih panjang, sehingga
keputusan pada dataset ini adalah mempertahankan nilai asli dan mencatatnya
sebagai temuan kualitas data.

Pada dataset Cosmopedia Indonesia, kolom `token_length` digunakan untuk mengukur panjang token pada setiap data. Nilai yang berada di luar batas IQR tetap dipertahankan karena belum tentu merupakan kesalahan data.

In [5]:
KOLOM_NUMERIK = "token_length"

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)

iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

jumlah_outlier = df_missing_handled.filter(
    (pl.col(KOLOM_NUMERIK) < batas_bawah)
    | (pl.col(KOLOM_NUMERIK) > batas_atas)
).height

print(f"Q1: {q1}")
print(f"Q3: {q3}")
print(f"IQR: {iqr}")
print(f"Batas bawah: {batas_bawah}")
print(f"Batas atas: {batas_atas}")
print(f"Jumlah outlier: {jumlah_outlier:,}")
print(
    f"Persentase outlier: "
    f"{jumlah_outlier / len(df_missing_handled) * 100:.2f}%"
)

# Outlier dipertahankan; tidak dilakukan clipping/winsorization.
df_outlier_handled = df_missing_handled

Q1: 1065.0
Q3: 1296.0
IQR: 231.0
Batas bawah: 718.5
Batas atas: 1642.5
Jumlah outlier: 3,790
Persentase outlier: 3.16%


## 5. Deduplikasi

Kolom `id` digunakan sebagai kunci unik dataset. Jika ditemukan ID berulang,
satu baris pertama dipertahankan agar setiap ID hanya muncul sekali.

Pada dataset Cosmopedia Indonesia, pemeriksaan duplikasi dilakukan berdasarkan kolom `id`. Langkah ini bertujuan memastikan setiap data memiliki identitas yang unik sebelum disimpan ke dalam format Parquet.

In [6]:
KOLOM_KUNCI = "id"

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique()

print(f"Total baris: {total_baris:,}")
print(f"ID unik: {baris_unik:,}")
print(f"Duplikat ID: {total_baris - baris_unik:,}")

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI],
    keep="first",
    maintain_order=True
)

print(f"Baris setelah deduplikasi: {len(df_dedup):,}")

Total baris: 120,068
ID unik: 120,068
Duplikat ID: 0
Baris setelah deduplikasi: 120,068


## 6. Validasi Data Dasar

Validasi dilakukan untuk memastikan kolom inti tidak kosong, ID tidak
terduplikasi, dan `token_length` bernilai positif. Null pada kolom metadata
tidak dianggap sebagai pelanggaran karena kolom tersebut bukan informasi inti
untuk membentuk pasangan prompt dan teks.

Pada dataset Cosmopedia Indonesia, validasi dilakukan menggunakan DuckDB untuk memeriksa kembali jumlah baris, kelengkapan kolom inti, nilai `token_length`, serta keunikan ID setelah proses cleaning.

In [7]:
connection.unregister("dataset")
connection.register("dataset_bersih", df_dedup)

query_validasi = (
    "SELECT "
    "COUNT(*) AS total_baris, "
    "SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS id_null, "
    "SUM(CASE WHEN prompt IS NULL OR TRIM(prompt) = '' THEN 1 ELSE 0 END) AS prompt_kosong, "
    "SUM(CASE WHEN text IS NULL OR TRIM(text) = '' THEN 1 ELSE 0 END) AS text_kosong, "
    "SUM(CASE WHEN token_length IS NULL OR token_length <= 0 THEN 1 ELSE 0 END) AS token_length_tidak_valid, "
    "COUNT(DISTINCT id) AS jumlah_id_unik "
    "FROM dataset_bersih"
)

validasi = connection.execute(query_validasi).fetchdf()

display(validasi)

,total_baris,id_null,prompt_kosong,text_kosong,token_length_tidak_valid,jumlah_id_unik
0,120068,0.0,0.0,0.0,0.0,120068


## 7. Simpan Data Bersih

Simpan hasil cleaning ke format Parquet pada folder `data/processed/`.
File ini digunakan sebagai sumber data Milestone 3.

Hasil pembersihan dataset Cosmopedia Indonesia disimpan dengan nama `cosmopedia_id_clean.parquet`. Format Parquet digunakan agar data dapat dibaca kembali secara efisien pada tahap analisis eksploratif di Milestone 3.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_dedup.write_parquet(
    OUTPUT_PATH,
    compression="zstd"
)

print(f"Baris sebelum cleaning: {len(df):,}")
print(f"Baris sesudah cleaning: {len(df_dedup):,}")
print(f"Kolom: {len(df_dedup.columns)}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")

Baris sebelum cleaning: 120,068
Baris sesudah cleaning: 120,068
Kolom: 9
Data bersih tersimpan di: /home/jovyan/work/tugas1-abd-2026/data/processed/cosmopedia_id_clean.parquet


## Catatan Milestone 2

Berdasarkan proses profiling dan cleaning pada dataset Cosmopedia Indonesia, diperoleh beberapa temuan terkait kualitas data:

1. **Dimensi kualitas data yang paling bermasalah**

   Permasalahan utama terdapat pada kelengkapan data (completeness), yaitu adanya nilai kosong pada kolom metadata `audience`, `format`, dan `seed_data`, masing-masing sebanyak satu nilai. Selain itu, ditemukan outlier pada kolom `token_length` yang menunjukkan adanya variasi panjang teks.

2. **Strategi penanganan missing values**

   Kolom inti `id`, `prompt`, dan `text` diperiksa sebagai informasi utama dataset. Baris yang memiliki nilai kosong pada kolom inti akan dikeluarkan. Sementara itu, nilai kosong pada kolom metadata dipertahankan sebagai null karena tidak tersedia informasi yang valid untuk mengisi nilai tersebut.

3. **Strategi penanganan outlier**

   Deteksi outlier dilakukan pada kolom `token_length` menggunakan metode Interquartile Range (IQR). Hasil pemeriksaan sebelumnya menunjukkan 3.790 outlier atau sekitar 3,16% dari data. Outlier dipertahankan karena nilai token yang tinggi belum tentu merupakan kesalahan pencatatan dan dapat merepresentasikan panjang teks yang sebenarnya.

4. **Jumlah baris yang berubah**

   Dataset awal memiliki 120.068 baris. Tidak ditemukan ID duplikat pada pemeriksaan sebelumnya. Jumlah akhir setelah penanganan missing values dan deduplikasi ditampilkan melalui hasil eksekusi notebook. Baris hanya akan berkurang apabila ditemukan nilai kosong pada kolom inti atau ID duplikat.

5. **Aturan validasi yang diterapkan**

   Validasi dilakukan menggunakan DuckDB untuk memastikan:
   - Kolom `id` tidak memiliki nilai kosong.
   - Kolom `prompt` dan `text` tidak kosong.
   - Kolom `token_length` memiliki nilai positif dan tidak kosong.
   - Jumlah ID unik sama dengan jumlah keseluruhan baris.
   - Nilai kosong pada kolom metadata diperbolehkan.

Hasil cleaning disimpan dalam format Parquet dengan nama `cosmopedia_id_clean.parquet` pada folder `data/processed/` untuk digunakan dalam Milestone 3.